# THIS CODE IS COPIED FROM "classification_modeler.ipynb" WITH SLIGHTLY REQUIRED MODIFICATION OF OUTPUT FILES AND CODE FOR CLASSIFICATION MODEL WITH SYNTHETIC DATA

In [ ]:
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split,StratifiedKFold
import joblib
import optuna
import pandas as pd
import numpy as np
import os

In [ ]:
def save_show_model_report(model,x,y,model_name):
    print(f"[++] PRINTING MODEL REPORT FOR MODEL : {model_name}")
    train_x,val_x,train_y,val_y = train_test_split(x,y,test_size=0.1)
    model.fit(train_x,train_y)
    pred_y = model.predict(val_x)
    acc_score = accuracy_score(val_y,pred_y)
    print(f"[++] PRINTING ACCURACY SCORES ....")
    print(f" accuracy_score:{acc_score}")
    print(f"[++] SAVING THE MODEL ")
    return acc_score
    
    

# UNCOMMENT THE MODEL YOU WANT TO RUN IN BELOW FUNCTION

In [ ]:
def objective(trial):
#     rf_clf = RandomForestClassifier(n_estimators=trial.suggest_categorical('rf_est',[100,200,300]),
#                                     criterion=trial.suggest_categorical('rf_crt',["gini","entropy"]),verbose=1)
#     score = save_show_model_report(rf_clf,x,y,"Random_Forest")
#     et_clf = ExtraTreesClassifier(n_estimators=trial.suggest_categorical('et_est',[100,200,300]),
#                                   class_weight=trial.suggest_categorical('et_cl_wt',["balanced", "balanced_subsample"]),verbose=1)
#     score = save_show_model_report(et_clf,x,y,"Extra_Trees")  
#     lr_clf = LogisticRegression(penalty="none",
#                                 C=trial.suggest_categorical('lr_C',[1.0,0.1]),
#                                 solver=trial.suggest_categorical('lr_solver',["newton-cg", "lbfgs", "sag", "saga"])
#                                 ,verbose=1)
#     score = save_show_model_report(lr_clf,x,y,"Logistic_Regression") 
#     Kn_clf = KNeighborsClassifier(n_neighbors=trial.suggest_categorical("k_nei",[3,4,5]),
#                                   weights=trial.suggest_categorical("k_wei",["uniform","distance"]),
#                                   algorithm=trial.suggest_categorical("K_algo",["auto","ball_tree","kd_tree","brute"]),
#                                  p=trial.suggest_categorical("K_p",[1,2]))
#     score = save_show_model_report(Kn_clf,x,y,"K_Neighbors")
    SVC_clf = SVC(kernel=trial.suggest_categorical("SVC_ker",["linear","poly","rbf","sigmoid"]),
                 gamma = trial.suggest_categorical("SVC_gamma",["scale","auto"]),
                 verbose=1)
    score = save_show_model_report(SVC_clf,x,y,"SVC")
#     DT_clf = DecisionTreeClassifier(criterion=trial.suggest_categorical("DT_cri",["gini","entropy"]),
#                                    splitter=trial.suggest_categorical("DT_spl",["best","random"]))
#     score = save_show_model_report(DT_clf,x,y,"Decision_Tree")
#     XGB_clf = XGBClassifier(n_estimators=trial.suggest_categorical("XGB_est",[100,200,300]),
#                            learning_rate = trial.suggest_categorical("XGB_lr",[0.1,0.01,0.001]),
#                            booster= trial.suggest_categorical("XGB_boost",["gbtree","gblinear","dart"]),
#                            tree_method = trial.suggest_categorical("XGB_tree",["exact","approx","hist","gpu_hist"]),
#                            predictor="cpu_predictor")
#     score = save_show_model_report(XGB_clf,x,y,"XGB_Classifier")  
    return score

In [ ]:
main_df = pd.read_csv("./main_datasets/smoted_scaled_efficiency_cat_data.csv")
y = main_df["target"]
main_df.drop("target",axis=1,inplace=True)
x = main_df

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=100)
print("[++] PRINTING BEST PARAMETERS")
best_p = study.best_params
print(best_p)

# MENTION MODEL NAME BELOW

In [ ]:
modelName = "SVC" #EDIT MODEL NAME HERE

In [ ]:
skf = StratifiedKFold(n_splits=10,shuffle=True,random_state=123)
acc_list = []
for num,(train_idx,test_idx) in enumerate(skf.split(x,y)):
    main_clf_model= SVC(kernel="linear",gamma="scale")      #change the model here acc to the objective function 
    x_train,x_test = x.loc[train_idx],x.loc[test_idx]
    y_train,y_test = y.loc[train_idx],y.loc[test_idx]
    main_clf_model.fit(x_train,y_train)
    score = main_clf_model.score(x_test,y_test)
    acc_list.append(score)
    print(score)
    print(f"[++] FOLD {num} DEPOSITED")
    try:
        joblib.dump(main_clf_model,f"./smoted_models/{modelName}_folded_models/{modelName}_fold{num}_tuned.pkl")
        print(f"[++] FOLD {num} model is deposited")
    except:
        os.mkdir(f"./smoted_models/{modelName}_folded_models")
        print(f"[++] FILE SYSTEM CREATED ")
        joblib.dump(main_clf_model,f"./smoted_models/{modelName}_folded_models/{modelName}_fold{num}_tuned.pkl")
        print(f"[++] FOLD {num} model is deposited")

In [ ]:
def mean(li):
    tot = 0
    for i in li:
        tot+=i
    m = tot/len(li)
    return m

In [ ]:

log = f'LOG FOR :{(modelName)} \n'
log += "BEST PARAMETERS :: "
log += str(best_p)
log += "\n"
log += "\n MODEL RESULTS\n"
log += f"AVERAGE ACCURACY : {mean(acc_list)}\n"
log += f"MAX ACCURACY : {max(acc_list)}\n"
log += f"MIN ACCURACY : {min(acc_list)}"

In [ ]:
print(log)

In [ ]:
with open(f"./smoted_models/{modelName}_folded_models/{modelName}_log.txt","w+") as file:
    file.write(log)
    file.close()